# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 2 — Final models, accuracy, calibration, conformal prediction and applicability domain

This notebook fits the final models on the primary split using the configurations selected
in Part 1, and evaluates them on three of the axes reported in the study: predictive
accuracy, probability calibration and the reliability of set-valued predictions. It also
characterises the applicability domain and establishes the structural-alert baseline
against which the learned models are compared.

Both families are seed-ensembled over the same three seeds, and every calibration
operation is applied identically to both, so that no part of the comparison depends on how
a family's outputs happen to be produced.

**Contents**

1. Environment and shared definitions
2. Data, partitions and representations
3. Final model fitting
4. Predictive accuracy
5. Probability calibration
6. Conformal prediction
7. Applicability domain
8. Structural-alert baseline
9. Persisted artefacts

Model weights and predictions are written to Drive for use in Part 3.

## 1. Environment and shared definitions

In [1]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q optuna
!pip install -q scikit-posthocs
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK    = "/content/drive/MyDrive/ames_pipeline"
DATADIR = f"{WORK}/data"
OUTDIR  = f"{WORK}/results"
CKPT    = f"{WORK}/ckpt"
FIGDIR  = f"{WORK}/figures"
MODELS  = f"{WORK}/models"
for d in (WORK, DATADIR, OUTDIR, CKPT, FIGDIR, MODELS):
    os.makedirs(d, exist_ok=True)
print("Working directory:", WORK)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ames_pipeline


The definitions shared by Parts 2 and 3 — featurisation, partitioning, the model classes
and the training routine — are written to a single module so that every stage of the study
uses one implementation.

In [3]:
%%writefile /content/ames_lib.py
"""Shared definitions for the Ames mutagenicity pipeline."""

import random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# ----------------------------- partitioning -----------------------------

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi


def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs  = list(groups.values())
    rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d)
    n_train, n_train_valid = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= n_train:
            tr += g
        elif len(tr) + len(va) + len(g) <= n_train_valid:
            va += g
        else:
            te += g
    return d.loc[tr], d.loc[va], d.loc[te]


def make_random_split(d, seed, frac=(0.8, 0.1, 0.1)):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(d.index.values)
    n = len(d)
    a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    return d.loc[idx[:a]], d.loc[idx[a:b]], d.loc[idx[b:]]


# ---------------------------- representations ----------------------------

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr


def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None:
            X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)


def one_hot(x, allowable):
    if x not in allowable:
        x = allowable[-1]
    return [int(x == a) for a in allowable]


def atom_features(atom):
    f  = one_hot(atom.GetSymbol(), ATOM_LIST)
    f += one_hot(str(atom.GetHybridization()), HYBRIDIZATION)
    f += [atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
          atom.GetTotalNumHs(), int(atom.IsInRing())]
    return f


def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]


def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        ei += [[i, j], [j, i]]
        ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)


def build_dataset(d):
    graphs = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [g for g in graphs if g is not None]


# ------------------------------ architectures ------------------------------

class GNN(nn.Module):
    """Message-passing classifier.

    GINE and GAT consume bond features; GCN operates on connectivity alone.
    Setting use_edge_attr=False replaces GINE with GIN, which allows the
    contribution of bond information to be isolated.
    """

    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn":
                conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else:
                raise ValueError(f"Unknown architecture: {kind}")
            self.convs.append(conv)
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, data):
        x, edge_index, edge_attr, batch = (data.x, data.edge_index,
                                           data.edge_attr, data.batch)
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = (conv(x, edge_index, edge_attr) if self.use_edge_attr
                 else conv(x, edge_index))
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, batch), global_add_pool(x, batch)], dim=1)
        return self.head(g).squeeze(-1)


def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4,
              use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads,
               use_edge_attr=use_edge_attr).to(DEVICE)


# ------------------------------- training -------------------------------

@torch.no_grad()
def evaluate(model, loader):
    """Return labels, predicted probabilities and logits."""
    model.eval()
    ys, ps, logits = [], [], []
    for d in loader:
        d = d.to(DEVICE)
        logit = model(d)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(logit).cpu().numpy())
        logits.append(logit.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(logits)


def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, trial=None, use_edge_attr=True):
    """Train one graph model; return it with its best validation ROC-AUC."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_loader = DataLoader(tr_set, batch_size=params["batch_size"],
                              shuffle=True, drop_last=True)
    valid_loader = DataLoader(va_set, batch_size=256)

    best_auc, best_state, waited = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for d in train_loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(d), d.y.view(-1)).backward()
            optimiser.step()
        y_va, p_va, _ = evaluate(model, valid_loader)
        auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else 0.5
        if trial is not None:
            trial.report(auc, epoch)
            if trial.should_prune():
                import optuna
                raise optuna.TrialPruned()
        if auc > best_auc:
            best_auc, waited = auc, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_auc

Writing /content/ames_lib.py


In [4]:
import sys, json, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames_lib import *
from torch_geometric.loader import DataLoader

print("Device:", DEVICE, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

SEED                 = 42
PRIMARY_SPLIT_SEED   = 101
SCAFFOLD_SPLIT_SEEDS = list(range(42, 57))
SEEDS                = [42, 43, 44]
MAX_EPOCHS, PATIENCE = 250, 30
assert PRIMARY_SPLIT_SEED not in SCAFFOLD_SPLIT_SEEDS

HP_CLASSIC = json.load(open(f"{OUTDIR}/hpo_classic.json"))
HP_GNN     = json.load(open(f"{OUTDIR}/hpo_gnn.json"))
for k, v in {**HP_CLASSIC, **HP_GNN}.items():
    print(f"  {k:14s} {v}")

Device: cuda | Tesla T4
  XGBoost        {'max_depth': 11, 'learning_rate': 0.15642823651322757, 'subsample': 0.6396975505413772, 'colsample_bytree': 0.5821188098787602, 'reg_lambda': 0.06842025683573061, 'reg_alpha': 0.03908769860097184, 'min_child_weight': 4}
  RandomForest   {'n_estimators': 900, 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 4, 'max_features': 'sqrt'}
  gcn            {'hidden': 256, 'n_layers': 3, 'dropout': 0.05358601947429411, 'lr': 0.004132189266590333, 'weight_decay': 4.705119022984706e-06, 'batch_size': 128, 'residual': True}
  gin            {'hidden': 64, 'n_layers': 4, 'dropout': 0.20731860936937893, 'lr': 0.0018017848240952697, 'weight_decay': 2.6566273102091933e-06, 'batch_size': 32, 'residual': False}
  gat            {'hidden': 128, 'n_layers': 3, 'dropout': 0.11678477971374823, 'lr': 0.00093642486119946, 'weight_decay': 0.0005566438039612761, 'batch_size': 32, 'residual': True}


## 2. Data, partitions and representations

The compound pool, the primary partition and both molecular representations are rebuilt
from the same seeds used in Part 1, so the split is identical without needing to be
carried between notebooks.

In [5]:
SOURCE_FILES = ["train.csv", "valid.csv", "test.csv"]

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in SOURCE_FILES],
               ignore_index=True)[["Drug_ID", "smiles", "label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)

df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]

X_train, df_train = build_X(df_train); y_train = df_train["label"].values
X_valid, df_valid = build_X(df_valid); y_valid = df_valid["label"].values
X_test,  df_test  = build_X(df_test);  y_test  = df_test["label"].values

train_set = build_dataset(df_train)
valid_set = build_dataset(df_valid)
test_set  = build_dataset(df_test)

valid_loader = DataLoader(valid_set, batch_size=256)
test_loader  = DataLoader(test_set,  batch_size=256)

pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                          dtype=torch.float, device=DEVICE)

print(f"Train {len(df_train)} | Valid {len(df_valid)} | Test {len(df_test)}")
print(f"Positive rate: train {100*y_train.mean():.1f}%, "
      f"valid {100*y_valid.mean():.1f}%, test {100*y_test.mean():.1f}%")

Train 5804 | Valid 725 | Test 726
Positive rate: train 55.6%, valid 53.7%, test 46.3%


## 3. Final model fitting

Each model is fitted three times under different random seeds and the ensemble prediction
is the mean of the three. The same seeds and the same ensembling rule are used for both
families. Trained graph weights are written to Drive so that the attribution analyses in
Part 3 operate on exactly these models.

In [6]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
classic_valid, classic_test, classic_models = {}, {}, {}

for name in ["XGBoost", "RandomForest"]:
    pv, pt, models = [], [], []
    for s in SEEDS:
        if name == "XGBoost":
            m = XGBClassifier(n_estimators=1500, **HP_CLASSIC["XGBoost"],
                              scale_pos_weight=scale_pos, eval_metric="auc",
                              early_stopping_rounds=50, tree_method="hist",
                              random_state=s, n_jobs=-1)
            m.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
        else:
            m = RandomForestClassifier(**HP_CLASSIC["RandomForest"],
                                       class_weight="balanced", random_state=s, n_jobs=-1)
            m.fit(X_train, y_train)
        pv.append(m.predict_proba(X_valid)[:, 1])
        pt.append(m.predict_proba(X_test)[:, 1])
        models.append(m)
    classic_valid[name] = np.mean(pv, axis=0)
    classic_test[name]  = np.mean(pt, axis=0)
    classic_models[name] = models
    print(f"{name:14s} fitted over {len(SEEDS)} seeds")

XGBoost        fitted over 3 seeds
RandomForest   fitted over 3 seeds


In [7]:
gnn_valid, gnn_test = {}, {}

for kind in ["gcn", "gin", "gat"]:
    pv, pt = [], []
    for s in SEEDS:
        path = f"{MODELS}/{kind}_seed{s}.pt"
        model, _ = train_gnn(kind, HP_GNN[kind], s, train_set, valid_set,
                             pos_weight, MAX_EPOCHS, PATIENCE)
        torch.save({"state_dict": model.state_dict(),
                    "params": HP_GNN[kind], "kind": kind, "seed": s}, path)
        _, p_va, _ = evaluate(model, valid_loader)
        _, p_te, _ = evaluate(model, test_loader)
        pv.append(p_va); pt.append(p_te)
        del model; torch.cuda.empty_cache()
    gnn_valid[kind.upper()] = np.mean(pv, axis=0)
    gnn_test[kind.upper()]  = np.mean(pt, axis=0)
    print(f"{kind.upper():5s} fitted over {len(SEEDS)} seeds, weights saved")

valid_prob = {**classic_valid, **gnn_valid}
test_prob  = {**classic_test,  **gnn_test}
MODEL_ORDER = ["XGBoost", "RandomForest", "GCN", "GIN", "GAT"]

GCN   fitted over 3 seeds, weights saved
GIN   fitted over 3 seeds, weights saved
GAT   fitted over 3 seeds, weights saved


## 4. Predictive accuracy

Threshold-dependent metrics use the decision threshold that maximises balanced accuracy on
the validation set, so that the operating point is chosen without reference to the test
data. Sensitivity and specificity are reported alongside balanced accuracy, since a
screening model is judged on both.

In [8]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, matthews_corrcoef,
                             recall_score, accuracy_score, f1_score)

def best_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    return grid[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int))
                               for t in grid]))]

rows, thresholds = [], {}
for m in MODEL_ORDER:
    thr = best_threshold(y_valid, valid_prob[m]); thresholds[m] = thr
    p = test_prob[m]; yhat = (p >= thr).astype(int)
    rows.append({
        "model":         m,
        "threshold":     thr,
        "ROC_AUC":       roc_auc_score(y_test, p),
        "PR_AUC":        average_precision_score(y_test, p),
        "accuracy":      accuracy_score(y_test, yhat),
        "balanced_acc":  balanced_accuracy_score(y_test, yhat),
        "sensitivity":   recall_score(y_test, yhat, pos_label=1),
        "specificity":   recall_score(y_test, yhat, pos_label=0),
        "F1":            f1_score(y_test, yhat),
        "MCC":           matthews_corrcoef(y_test, yhat),
    })

acc_df = pd.DataFrame(rows)
acc_df.to_csv(f"{OUTDIR}/table01_accuracy.csv", index=False)
print(acc_df.round(4).to_string(index=False))

       model  threshold  ROC_AUC  PR_AUC  accuracy  balanced_acc  sensitivity  specificity     F1    MCC
     XGBoost       0.53   0.7824  0.7690    0.7231        0.7198       0.6756       0.7641 0.6931 0.4417
RandomForest       0.59   0.7994  0.7856    0.7397        0.7319       0.6280       0.8359 0.6907 0.4768
         GCN       0.52   0.7877  0.7445    0.7287        0.7277       0.7143       0.7410 0.7090 0.4549
         GIN       0.57   0.8133  0.7663    0.7507        0.7488       0.7232       0.7744 0.7286 0.4981
         GAT       0.53   0.7977  0.7799    0.7410        0.7402       0.7292       0.7513 0.7227 0.4799


## 5. Probability calibration

Every model is calibrated in the same way. Ensemble probabilities are mapped to logits, a
single temperature is fitted on the validation set by minimising negative log-likelihood,
and the fitted temperature is applied to the test logits. A temperature above one indicates
overconfidence.

Expected calibration error is reported at four equal-width binnings and, as adaptive
calibration error, at an equal-mass binning, since the ranking of models under this metric
can depend on how the probability axis is divided.

In [9]:
def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def fit_temperature(logits, y):
    z = torch.tensor(logits, dtype=torch.float32)
    t = torch.tensor(y, dtype=torch.float32)
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=200)
    lossf = torch.nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad()
        loss = lossf(z / torch.exp(log_T), t)
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.exp(log_T).item())

def ece_equal_width(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e

def ace_equal_mass(y, p, n_bins=10):
    order = np.argsort(p)
    e = 0.0
    for chunk in np.array_split(order, n_bins):
        if len(chunk) == 0:
            continue
        e += (len(chunk) / len(p)) * abs(y[chunk].mean() - p[chunk].mean())
    return e

rows, test_prob_cal = [], {}
for m in MODEL_ORDER:
    T = fit_temperature(to_logit(valid_prob[m]), y_valid)
    p_raw = test_prob[m]
    p_cal = 1 / (1 + np.exp(-to_logit(p_raw) / T))
    test_prob_cal[m] = p_cal
    row = {"model": m, "temperature": T,
           "ECE_raw": ece_equal_width(y_test, p_raw, 10),
           "ECE_calibrated": ece_equal_width(y_test, p_cal, 10),
           "ACE_raw": ace_equal_mass(y_test, p_raw, 10)}
    for nb in (5, 10, 15, 20):
        row[f"ECE_{nb}bins"] = ece_equal_width(y_test, p_raw, nb)
    rows.append(row)

cal_df = pd.DataFrame(rows)
cal_df.to_csv(f"{OUTDIR}/table02_calibration.csv", index=False)
print(cal_df.round(4).to_string(index=False))

       model  temperature  ECE_raw  ECE_calibrated  ACE_raw  ECE_5bins  ECE_10bins  ECE_15bins  ECE_20bins
     XGBoost       0.7701   0.0586          0.0736   0.0596     0.0586      0.0586      0.0640      0.0608
RandomForest       0.6961   0.0906          0.0786   0.0759     0.0890      0.0906      0.0929      0.0958
         GCN       1.1078   0.0750          0.0635   0.0678     0.0750      0.0750      0.0825      0.0851
         GIN       1.1451   0.0821          0.0678   0.0835     0.0802      0.0821      0.0837      0.0947
         GAT       1.4645   0.0996          0.0676   0.0994     0.0951      0.0996      0.1016      0.1029


## 6. Conformal prediction

An inductive conformal predictor is calibrated on the validation set with the
nonconformity score 1 - p(y), and applied to the test set at a nominal error rate of 0.10.
Empirical coverage is reported with an exact binomial confidence interval, so that
deviations from the nominal level can be distinguished from sampling variation on a test
set of this size.

In [10]:
from scipy import stats

ALPHA = 0.10
rows = []
for m in MODEL_ORDER:
    pv, pt = valid_prob[m], test_prob[m]
    cal_scores = np.where(y_valid == 1, 1 - pv, pv)
    n = len(cal_scores)
    q = np.quantile(cal_scores, min(1.0, np.ceil((n + 1) * (1 - ALPHA)) / n),
                    method="higher")
    inc_pos = (1 - pt) <= q
    inc_neg = pt <= q
    covered = np.where(y_test == 1, inc_pos, inc_neg)
    set_size = inc_pos.astype(int) + inc_neg.astype(int)

    k = int(covered.sum()); N = len(covered)
    lo = stats.beta.ppf(0.025, k, N - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(0.975, k + 1, N - k) if k < N else 1.0
    p_bin = stats.binomtest(k, N, 1 - ALPHA).pvalue

    rows.append({
        "model": m, "nominal_coverage": 1 - ALPHA,
        "coverage": covered.mean(), "n_test": N,
        "coverage_ci_low": lo, "coverage_ci_high": hi,
        "p_vs_nominal": p_bin,
        "undercoverage_resolvable": bool(hi < (1 - ALPHA)),
        "avg_set_size": set_size.mean(),
        "singleton_rate": (set_size == 1).mean(),
        "empty_rate": (set_size == 0).mean(),
    })

conf_df = pd.DataFrame(rows)
conf_df.to_csv(f"{OUTDIR}/table03_conformal.csv", index=False)
print(conf_df.round(4).to_string(index=False))

       model  nominal_coverage  coverage  n_test  coverage_ci_low  coverage_ci_high  p_vs_nominal  undercoverage_resolvable  avg_set_size  singleton_rate  empty_rate
     XGBoost               0.9    0.8912     726           0.8662            0.9129        0.4212                     False        1.4394          0.5606         0.0
RandomForest               0.9    0.8857     726           0.8603            0.9079        0.1941                     False        1.4063          0.5937         0.0
         GCN               0.9    0.8595     726           0.8321            0.8840        0.0005                      True        1.3196          0.6804         0.0
         GIN               0.9    0.8760     726           0.8498            0.8991        0.0352                      True        1.3361          0.6639         0.0
         GAT               0.9    0.8760     726           0.8498            0.8991        0.0352                      True        1.3485          0.6515         0.0


## 7. Applicability domain

Each test compound is assigned the maximum Tanimoto similarity to any training compound.
Performance inside and outside the domain is compared across a range of thresholds, so
that the conclusion does not rest on one arbitrary cut-off. A paired bootstrap over test
compounds tests whether the two model families degrade differently outside the domain.

In [11]:
from rdkit import DataStructs as DS

train_fps = [morgan_gen.GetFingerprint(Chem.MolFromSmiles(s))
             for s in df_train["canonical"]]
nn_sim = np.array([max(DS.BulkTanimotoSimilarity(
                       morgan_gen.GetFingerprint(Chem.MolFromSmiles(s)), train_fps))
                   for s in df_test["canonical"]])
np.save(f"{OUTDIR}/ad_nn_similarity.npy", nn_sim)
print(f"Nearest-neighbour similarity: median {np.median(nn_sim):.3f}, "
      f"IQR [{np.percentile(nn_sim,25):.3f}, {np.percentile(nn_sim,75):.3f}]")

def safe_auc(y, p):
    return roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan

rows = []
for thr in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    inside = nn_sim >= thr
    for m in MODEL_ORDER:
        a_in  = safe_auc(y_test[inside],  test_prob[m][inside])
        a_out = safe_auc(y_test[~inside], test_prob[m][~inside])
        rows.append({"threshold": thr, "model": m,
                     "n_inside": int(inside.sum()), "n_outside": int((~inside).sum()),
                     "auc_inside": a_in, "auc_outside": a_out, "delta": a_out - a_in})
ad_df = pd.DataFrame(rows)
ad_df.to_csv(f"{OUTDIR}/table04_applicability_domain.csv", index=False)
print()
print(ad_df.pivot_table(index="threshold", columns="model", values="delta")
          .reindex(columns=MODEL_ORDER).round(3).to_string())

# Family comparison at the reference threshold, by paired bootstrap over compounds.
THR = 0.30
CLASSICAL, GRAPH = ["XGBoost", "RandomForest"], ["GCN", "GIN", "GAT"]
rng = np.random.default_rng(SEED)
diffs = []
for _ in range(2000):
    idx = rng.integers(0, len(y_test), len(y_test))
    ys, sim_b = y_test[idx], nn_sim[idx]
    ins = sim_b >= THR
    if len(np.unique(ys[ins])) < 2 or len(np.unique(ys[~ins])) < 2:
        continue
    d = {m: safe_auc(ys[~ins], test_prob[m][idx][~ins])
            - safe_auc(ys[ins],  test_prob[m][idx][ins]) for m in MODEL_ORDER}
    diffs.append(np.mean([d[m] for m in CLASSICAL]) - np.mean([d[m] for m in GRAPH]))
diffs = np.array(diffs)
fam = {"threshold": THR, "n_bootstrap": len(diffs),
       "mean_classical_minus_graph": diffs.mean(),
       "ci_low": np.percentile(diffs, 2.5), "ci_high": np.percentile(diffs, 97.5),
       "p_two_sided": 2 * min((diffs > 0).mean(), (diffs < 0).mean())}
pd.DataFrame([fam]).to_csv(f"{OUTDIR}/table04b_ad_family_test.csv", index=False)
print()
for k, v in fam.items():
    print(f"  {k:28s}: {v if isinstance(v, int) else round(float(v), 4)}")

Nearest-neighbour similarity: median 0.449, IQR [0.327, 0.571]

model      XGBoost  RandomForest    GCN    GIN    GAT
threshold                                            
0.20        -0.350        -0.409 -0.251 -0.171 -0.285
0.25        -0.284        -0.333 -0.122 -0.146 -0.196
0.30        -0.144        -0.187 -0.128 -0.084 -0.140
0.35        -0.170        -0.208 -0.147 -0.114 -0.140
0.40        -0.122        -0.155 -0.118 -0.067 -0.100
0.50        -0.084        -0.089 -0.061 -0.022 -0.049

  threshold                   : 0.3
  n_bootstrap                 : 2000
  mean_classical_minus_graph  : -0.0476
  ci_low                      : -0.1455
  ci_high                     : 0.0429
  p_two_sided                 : 0.352


## 8. Structural-alert baseline

A rule-based classifier that simply counts matches to the toxicophore library provides a
reference point for how much of the endpoint is captured without learning. Its performance
also bounds how informative agreement with those alerts can be as a measure of explanation
quality.

In [12]:
TOXICOPHORES = {
    "aromatic_nitro":         "[a][N+](=O)[O-]",
    "aromatic_amine":         "[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":       "[a][NX2]=[OX1]",
    "alkyl_nitrite":          "[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":            "[NX3][NX2]=[OX1]",
    "epoxide":                "C1OC1",
    "aziridine":              "C1NC1",
    "azide":                  "[NX2]=[NX2+]=[NX1-]",
    "azo_aromatic":           "[a][NX2]=[NX2][a]",
    "aliphatic_halide":       "[CX4][F,Cl,Br,I]",
    "hetero_hetero_bond":     "[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":    "c1ccc2ccccc2c1",
    "aromatic_hydroxylamine": "[a][NX3;H1][OX2;H1]",
    "michael_acceptor":       "[CX3]=[CX3][CX3]=[OX1]",
    "aromatic_azoxy":         "[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":       "[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":     "[a][NX3;H1][NX3;H2]",
    "alkyl_sulfonate":        "[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
TOX_PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}
assert all(p is not None for p in TOX_PATTERNS.values())

def alert_count(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return 0
    return sum(1 for p in TOX_PATTERNS.values() if mol.HasSubstructMatch(p))

alerts_test = np.array([alert_count(s) for s in df_test["canonical"]])
yhat = (alerts_test >= 1).astype(int)
alerts_row = {
    "n_alerts":           len(TOX_PATTERNS),
    "ROC_AUC":            roc_auc_score(y_test, alerts_test),
    "accuracy":           accuracy_score(y_test, yhat),
    "balanced_acc":       balanced_accuracy_score(y_test, yhat),
    "sensitivity":        recall_score(y_test, yhat, pos_label=1),
    "specificity":        recall_score(y_test, yhat, pos_label=0),
    "MCC":                matthews_corrcoef(y_test, yhat),
    "molecules_with_alert": yhat.mean(),
}
pd.DataFrame([alerts_row]).to_csv(f"{OUTDIR}/table05_alerts_baseline.csv", index=False)
for k, v in alerts_row.items():
    print(f"  {k:22s}: {round(float(v), 4)}")

  n_alerts              : 18.0
  ROC_AUC               : 0.6779
  accuracy              : 0.6488
  balanced_acc          : 0.6554
  sensitivity           : 0.744
  specificity           : 0.5667
  MCC                   : 0.3136
  molecules_with_alert  : 0.5771


## 9. Persisted artefacts

Predictions and partition indices are stored so that Part 3 reproduces exactly the model
outputs evaluated here.

In [13]:
np.savez(f"{OUTDIR}/predictions_primary.npz",
         y_valid=y_valid, y_test=y_test, nn_sim=nn_sim,
         alerts_test=alerts_test,
         **{f"valid_{m}": valid_prob[m] for m in MODEL_ORDER},
         **{f"test_{m}":  test_prob[m]  for m in MODEL_ORDER},
         **{f"testcal_{m}": test_prob_cal[m] for m in MODEL_ORDER})

for name, d in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    d.to_csv(f"{OUTDIR}/primary_{name}.csv", index=False)

json.dump(thresholds, open(f"{OUTDIR}/decision_thresholds.json", "w"), indent=2)

print("Written to", OUTDIR)
for f in ["table01_accuracy.csv", "table02_calibration.csv", "table03_conformal.csv",
          "table04_applicability_domain.csv", "table04b_ad_family_test.csv",
          "table05_alerts_baseline.csv", "predictions_primary.npz",
          "ad_nn_similarity.npy", "decision_thresholds.json"]:
    print("  -", f)
print("Graph model weights written to", MODELS)

Written to /content/drive/MyDrive/ames_pipeline/results
  - table01_accuracy.csv
  - table02_calibration.csv
  - table03_conformal.csv
  - table04_applicability_domain.csv
  - table04b_ad_family_test.csv
  - table05_alerts_baseline.csv
  - predictions_primary.npz
  - ad_nn_similarity.npy
  - decision_thresholds.json
Graph model weights written to /content/drive/MyDrive/ames_pipeline/models


In [14]:
!cd /content/drive/MyDrive/ames_pipeline && zip -qr /content/part2_results.zip results
from google.colab import files
files.download("/content/part2_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>